# E0B external divergence

In [ ]:

from __future__ import annotations
import os,re,json,hashlib,subprocess,sys,math
from pathlib import Path, PurePosixPath
from collections import defaultdict
from typing import Any
import numpy as np,pandas as pd,matplotlib.pyplot as plt

RUN_MODE="debug"
USE_PUBLIC_CORPUS=True
env_root=os.environ.get("CERA_ROOT")
candidates=[Path(env_root)] if env_root else []
candidates += [Path("/kaggle/working/cera"),Path("/mnt/data/cera"),Path.cwd()]
ROOT=next((p for p in candidates if p.exists() and (p.name=="cera" or (p/"configs").exists())),Path.cwd())
LOCAL_ROOT=ROOT/"data"/"osworld"
RESULTS=ROOT/"results"/"E0B"; FIGURES=ROOT/"figures"/"E0B"
TARGET_PAIRS=3 if RUN_MODE=="debug" else 30
MIN_ACTIONS=5; MIN_EFFECTS=2; RANDOM_SEED=20260914
HF_REPO_ID="simular-ai/sai-osworld-v2-benchmark-runs"; HF_REVISION="main"
LOCAL_ROOT.mkdir(parents=True,exist_ok=True); RESULTS.mkdir(parents=True,exist_ok=True); FIGURES.mkdir(parents=True,exist_ok=True)
if not USE_PUBLIC_CORPUS: pass


In [ ]:

def ensure_package(import_name,pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pip_name or import_name])
import importlib.util
if USE_PUBLIC_CORPUS:
    ensure_package("huggingface_hub")
    from huggingface_hub import HfApi,hf_hub_download

def sha256_file(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
def load_jsonl(path):
    rows=[]
    with path.open("r",encoding="utf-8",errors="replace") as f:
        for n,line in enumerate(f,1):
            line=line.strip()
            if not line: continue
            try:o=json.loads(line)
            except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON {path}:{n}") from e
            if not isinstance(o,dict): raise ValueError(f"Non-object JSON {path}:{n}")
            rows.append(o)
    return rows
def parse_public_path(repo_path):
    parts=PurePosixPath(repo_path).parts
    if not parts or parts[-1]!="traj.jsonl" or "tasks" not in parts: return None
    run=next((x for x in parts if x in {"run1","run2"}),None)
    if run is None:return None
    i=parts.index("tasks")
    if i+1>=len(parts):return None
    task_id=parts[i+1]
    return (run,task_id) if re.fullmatch(r"\d{3}",task_id) else None
def discover_local(root):
    out={}
    for p in root.rglob("traj.jsonl"):
        k=parse_public_path(p.as_posix())
        if k: out[k]=p
    return out
def build_public_index():
    files=HfApi().list_repo_files(repo_id=HF_REPO_ID,repo_type="dataset",revision=HF_REVISION)
    return {k:f for f in files if (k:=parse_public_path(f)) is not None}
def download_pairs(task_ids,index):
    for tid in task_ids:
        for run in ("run1","run2"):
            repo=index.get((run,tid))
            if repo is None: continue
            target=LOCAL_ROOT/repo
            if target.exists(): continue
            target.parent.mkdir(parents=True,exist_ok=True)
            p=hf_hub_download(repo_id=HF_REPO_ID,filename=repo,repo_type="dataset",revision=HF_REVISION,local_dir=str(LOCAL_ROOT))
            if not Path(p).exists(): raise FileNotFoundError(p)


In [ ]:

SPACE_RE=re.compile(r"\s+")
SECRET_RE=re.compile(r"(?i)\b(?:password|token|secret|api[_ -]?key)\b\s*[:=]\s*[^\s,;]+")
TIMESTAMP_RE=re.compile(r"\b\d{8}@\d{9,}\b")
COORD_RE=re.compile(r"\b\d{1,4}\s*,\s*\d{1,4}\b")
URL_RE=re.compile(r'https?://[^\s]+',re.I)
FILE_RE=re.compile(r"(?:/home/[^\s]+|[A-Za-z]:\\[^\s]+|[\w./-]+\.(?:txt|csv|json|py|md|pdf|docx|xlsx|pptx|zip|tar|gz|html|xml))",re.I)
PACKAGE_RE=re.compile(r"\b(?:pip|pip3|apt|apt-get|npm|yarn|conda)\s+(?:install|remove|uninstall)\s+([\w.@+:-]+)",re.I)
GIT_RE=re.compile(r"\bgit\s+(commit|push|reset|revert|merge)\b",re.I)
EFFECT_VERBS=re.compile(r"\b(?:create|created|write|written|save|saved|delete|deleted|remove|removed|rename|renamed|move|moved|copy|copied|export|exported|download|downloaded|upload|uploaded|install|installed|uninstall|uninstalled|commit|committed|push|pushed|reset|revert|merge|submit|submitted|publish|published|deploy|deployed|send|sent|post|posted|apply|applied|approve|approved|book|booked|purchase|purchased|change|changed|update|updated|modify|modified|configure|configured)\b",re.I)
def normalize_text(v):
    s=str(v or ""); s=SECRET_RE.sub("<secret>",s); s=TIMESTAMP_RE.sub("<time>",s); s=COORD_RE.sub("<coord>",s); s=s.replace("\\","/"); s=SPACE_RE.sub(" ",s).strip().lower(); return s[:500]
def extract_effects(steps):
    out=[]
    for idx,step in enumerate(steps,1):
        action=str(step.get("action","")); response=str(step.get("response","")); text=f"{action}\n{response}"; norm=normalize_text(text)
        if not EFFECT_VERBS.search(text): continue
        step_num=int(step.get("step_num",idx) or idx)
        fm=FILE_RE.search(text); um=URL_RE.search(text); pm=PACKAGE_RE.search(text); gm=GIT_RE.search(text)
        if fm:key=f"file:{normalize_text(fm.group(0)).strip('.,:;()[]{}')}"; kind="file"
        elif um:key=f"url:{normalize_text(um.group(0)).rstrip('.,);]')}"; kind="url"
        elif pm:key=f"package:{normalize_text(pm.group(1))}"; kind="package"
        elif gm:key=f"git:{normalize_text(gm.group(1))}"; kind="git"
        elif re.search(r"\b(?:config|configuration|setting|settings|preference)\b",norm): key="config"; kind="config"
        elif re.search(r"\b(?:submit|publish|deploy|send|post|approve|book|purchase|apply)\b",norm): key=re.sub(r"[^a-z0-9_]+","_",norm)[:160]; kind="external_action"
        else: continue
        out.append({"step":step_num,"kind":kind,"key":key,"state":norm,"evidence":norm})
    unique={}
    for e in out: unique[(e["kind"],e["key"],e["state"])]=e
    return list(unique.values())
def action_family(action):
    a=normalize_text(action)
    if "doubleclick" in a or "double_click" in a:return "double_click"
    if "click" in a:return "click"
    if "write" in a or "typewrite" in a:return "write"
    if "press" in a or "hotkey" in a:return "key"
    if "scroll" in a:return "scroll"
    if "drag" in a or "move" in a:return "drag"
    if "shell" in a or "terminal" in a or "command" in a:return "shell"
    if "wait" in a or "sleep" in a:return "wait"
    return a[:40]
def action_signature(steps): return [action_family(str(s.get("action",""))) for s in steps]
def normalized_edit_distance(a,b):
    if not a and not b:return 0.0
    if not a or not b:return 1.0
    prev=list(range(len(b)+1))
    for i,ai in enumerate(a,1):
        cur=[i]
        for j,bj in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ai!=bj)))
        prev=cur
    return prev[-1]/max(len(a),len(b),1)


In [ ]:

public_index=build_public_index() if USE_PUBLIC_CORPUS else {}
common=sorted({t for r,t in public_index if r=="run1"} & {t for r,t in public_index if r=="run2"}) if USE_PUBLIC_CORPUS else sorted({t for r,t in discover_local(LOCAL_ROOT) if r=="run1"} & {t for r,t in discover_local(LOCAL_ROOT) if r=="run2"})
rng=np.random.default_rng(RANDOM_SEED)
shuffled=list(common); rng.shuffle(shuffled)

def get_paths(tid):
    if USE_PUBLIC_CORPUS:
        download_pairs([tid],public_index)
    idx=discover_local(LOCAL_ROOT); p1=idx.get(("run1",tid)); p2=idx.get(("run2",tid))
    return {"run1":p1,"run2":p2} if p1 and p2 and p1.exists() and p2.exists() else None

def qualifies(tid):
    pair=get_paths(tid)
    if pair is None:return False
    return all(len(load_jsonl(pair[r]))>=MIN_ACTIONS and len(extract_effects(load_jsonl(pair[r])))>=MIN_EFFECTS for r in ("run1","run2"))

selected=[]; rejected=[]
for tid in shuffled:
    if len(selected)>=TARGET_PAIRS: break
    if qualifies(tid): selected.append(tid)
    else: rejected.append(tid)
if len(selected)!=TARGET_PAIRS: raise RuntimeError(f"Only {len(selected)} qualified pairs; required {TARGET_PAIRS}.")
pairs=[{"task_id":tid,**get_paths(tid)} for tid in sorted(selected)]
print(f"Candidate common tasks: {len(common)}")
print(f"Trace-quality rejections before selection: {len(rejected)}")
print(f"Selected independent pair cases: {len(pairs)}")


In [ ]:

def build_record(pair):
    s1=load_jsonl(pair["run1"]); s2=load_jsonl(pair["run2"]); e1=extract_effects(s1); e2=extract_effects(s2)
    if len(s1)<MIN_ACTIONS or len(s2)<MIN_ACTIONS or len(e1)<MIN_EFFECTS or len(e2)<MIN_EFFECTS: raise RuntimeError(pair["task_id"])
    return {"task_id":pair["task_id"],"run1_path":str(pair["run1"]),"run2_path":str(pair["run2"]),
            "n_actions_1":len(s1),"n_actions_2":len(s2),"n_effects_1":len(e1),"n_effects_2":len(e2),
            "actions_1":action_signature(s1),"actions_2":action_signature(s2),"effects_1":e1,"effects_2":e2,
            "sha256_1":sha256_file(pair["run1"]),"sha256_2":sha256_file(pair["run2"])}
records=[build_record(p) for p in pairs]

def compare_effects(e1,e2):
    a=defaultdict(list); b=defaultdict(list)
    for e in e1:a[e["key"]].append(e)
    for e in e2:b[e["key"]].append(e)
    rows=[]
    for key in sorted(set(a)|set(b)):
        left,right=a.get(key,[]),b.get(key,[])
        s1={x["state"] for x in left}; s2={x["state"] for x in right}
        if left and right and s1&s2: status="preserved"
        elif left and right: status="contradicted"
        else: status="unknown"
        sample=(left or right)[0]
        rows.append({"kind":sample["kind"],"key":key,"status":status,"run1_present":bool(left),"run2_present":bool(right),
                     "run1_evidence":" || ".join(sorted(x["evidence"] for x in left))[:1000],
                     "run2_evidence":" || ".join(sorted(x["evidence"] for x in right))[:1000]})
    return rows

analyses=[]
for rec in records:
    rows=compare_effects(rec["effects_1"],rec["effects_2"]); st=[r["status"] for r in rows]
    p=sum(x=="preserved" for x in st); c=sum(x=="contradicted" for x in st); u=sum(x=="unknown" for x in st); obs=p+c
    analyses.append({k:v for k,v in {"task_id":rec["task_id"],"action_divergence":normalized_edit_distance(rec["actions_1"],rec["actions_2"]),
        "n_actions_1":rec["n_actions_1"],"n_actions_2":rec["n_actions_2"],"n_effects_1":rec["n_effects_1"],"n_effects_2":rec["n_effects_2"],
        "n_effect_keys":len(rows),"preserved_effects":p,"contradicted_effects":c,"unknown_effects":u,"observed_effects":obs,
        "mixed_compatibility":bool(p>0 and c>0),"adoption_opportunity_proxy":p/obs if obs else np.nan,
        "run1_path":rec["run1_path"],"run2_path":rec["run2_path"],"effect_rows":rows}.items()})
    print(f"[{len(analyses)}/{len(records)}] {rec['task_id']} div={analyses[-1]['action_divergence']:.3f}",flush=True)
df=pd.DataFrame([{k:v for k,v in a.items() if k!="effect_rows"} for a in analyses])
rows=[]
for a in analyses:
    for e in a["effect_rows"]: rows.append({"task_id":a["task_id"],**e})
effects_df=pd.DataFrame(rows)


In [ ]:

def wilson(successes,n,z=1.96):
    if n==0:return (np.nan,np.nan)
    p=successes/n; d=1+z*z/n; center=(p+z*z/(2*n))/d; half=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d; return center-half,center+half
gates={
    "target_pair_count":len(df)==TARGET_PAIRS,
    "unique_task_pairs":df.task_id.nunique()==len(df),
    "both_paths_exist":all(Path(x).exists() for x in df.run1_path)&all(Path(x).exists() for x in df.run2_path),
    "finite_divergence":bool(np.isfinite(df.action_divergence).all()),
    "nonnegative_counts":bool((df[["n_effect_keys","preserved_effects","contradicted_effects","unknown_effects"]]>=0).all().all()),
    "valid_status_labels":bool(effects_df.status.isin({"preserved","contradicted","unknown"}).all()),
    "mixed_definition_consistent":all(row.mixed_compatibility==(row.preserved_effects>0 and row.contradicted_effects>0) for _,row in df.iterrows())
}
if not all(gates.values()): raise RuntimeError("E0B implementation gate failed.")
scientific={"material_divergence_observed":bool((df.action_divergence>=0.25).any()),
            "preserved_effect_observed":bool((df.preserved_effects>0).any()),
            "contradicted_effect_observed":bool((df.contradicted_effects>0).any()),
            "mixed_compatibility_observed":bool(df.mixed_compatibility.any())}
summary={"paired_tasks":len(df),"any_divergence_rate":float((df.action_divergence>0).mean()),
         "material_divergence_rate":float((df.action_divergence>=0.25).mean()),
         "material_divergence_wilson":wilson(int((df.action_divergence>=0.25).sum()),len(df)),
         "mixed_compatibility_proxy_rate":float(df.mixed_compatibility.mean()),
         "mean_preserved_effects":float(df.preserved_effects.mean()),
         "mean_contradicted_effects":float(df.contradicted_effects.mean()),
         "mean_unknown_effects":float(df.unknown_effects.mean()),
         "pooled_observed_effect_preservation_rate":float(df.preserved_effects.sum()/max(df.observed_effects.sum(),1)),
         "mean_adoption_opportunity_proxy":float(df.adoption_opportunity_proxy.dropna().mean()) if df.adoption_opportunity_proxy.notna().any() else None,
         "scientific_signal":scientific}
df.to_csv(RESULTS/"e0b_pair_results.csv",index=False); effects_df.to_csv(RESULTS/"e0b_effect_results.csv",index=False)
(RESULTS/"e0b_manifest.json").write_text(json.dumps({"tasks":selected,"seed":RANDOM_SEED,"min_actions":MIN_ACTIONS,"min_effects":MIN_EFFECTS,"repo_id":HF_REPO_ID,"revision":HF_REVISION},indent=2),encoding="utf-8")
(RESULTS/"e0b_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
(RESULTS/"e0b_validation.json").write_text(json.dumps({"implementation_gates":gates,"scientific_signal":scientific},indent=2),encoding="utf-8")


In [ ]:

fig,ax=plt.subplots(figsize=(7,4.5)); ax.hist(df.action_divergence,bins=max(3,min(10,len(df))),edgecolor="black"); ax.set_xlabel("Normalized action divergence"); ax.set_ylabel("Task pairs"); fig.tight_layout(); fig.savefig(FIGURES/"e0b_action_divergence.png",dpi=180,bbox_inches="tight"); plt.show()
status_counts=effects_df.status.value_counts().reindex(["preserved","contradicted","unknown"],fill_value=0)
fig,ax=plt.subplots(figsize=(6.5,4.2)); ax.bar(status_counts.index,status_counts.values); ax.set_ylabel("Effect keys"); fig.tight_layout(); fig.savefig(FIGURES/"e0b_effect_status.png",dpi=180,bbox_inches="tight"); plt.show()
print(json.dumps(summary,indent=2))
